In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

# FLenQA J-Lens data-mining asset

This notebook builds the reusable matched short/long coefficient table. It uses the saved FLenQA full-run prompts, final positions, top-k directions, and scored model outputs. Coefficients use the existing `jlens_vector` and `lens_coordinates` definitions. No analysis is performed here.

In [ ]:
import numpy as np

import jlens
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
from datasets import load_from_disk
from jlens.hooks import ActivationRecorder
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.benchmarks.flenqa.positions import prepare_prompt
from jlens_reasoning.evaluation import evaluate_paper_binary
from jlens_reasoning.experiments_utils.interventions import jlens_vector, lens_coordinates
from jlens_reasoning.experiments_utils.validation import validate_model_lens

SHORT_CTX_SIZE, LONG_CTX_SIZE = 250, 1000
EXPECTED_PAIR_COUNT = 300
DIRECTION_TOP_K = 250
MAX_SEQ_LEN = 4096
FULL_RUN = context.runs_dir / "flenqa-full-run"
ACCURACY_PATH = context.runs_dir / "flenqa-accuracy" / "results.parquet"
OUTPUT_DIR = context.runs_dir / "flenqa-data-mining-asset"
ASSET_PATH = OUTPUT_DIR / "matched_jlens_coefficients.parquet"
SUMMARY_PATH = OUTPUT_DIR / "transition_summary.csv"

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
rows = normalize_rows(dataset["eval"] if hasattr(dataset, "keys") else dataset)
all_prompts = prepare_prompts(rows)

def nominal_context_size(prompt):
    sizes = {item.ctx_size for item in prompt.provenance}
    if len(sizes) != 1:
        raise ValueError(f"Prompt {prompt.prompt_id} spans context sizes {sizes}")
    return sizes.pop()

by_problem = {}
for prompt in all_prompts:
    by_problem.setdefault(prompt.problem_id, []).append(prompt)
pairs = []
for problem_id, candidates in sorted(by_problem.items()):
    short_candidates = [p for p in candidates if nominal_context_size(p) == SHORT_CTX_SIZE]
    long_candidates = [p for p in candidates if nominal_context_size(p) == LONG_CTX_SIZE]
    if not short_candidates or not long_candidates:
        continue
    short = min(short_candidates, key=lambda p: p.canonical_index)
    long = min(long_candidates, key=lambda p: p.canonical_index)
    assert short.problem_id == long.problem_id == problem_id
    assert (short.task, short.label, short.question) == (long.task, long.label, long.question)
    pairs.append({"problem_id": problem_id, "short": short, "long": long})
assert len(pairs) == EXPECTED_PAIR_COUNT, len(pairs)

accuracy = pd.read_parquet(ACCURACY_PATH)
required_accuracy_columns = {"prompt_id", "problem_id", "ctx_size", "correct", "generated_text"}
assert required_accuracy_columns <= set(accuracy.columns)
accuracy_by_id = accuracy.set_index("prompt_id")
assert accuracy_by_id.index.is_unique

positions = pd.read_parquet(FULL_RUN / "positions")
final_positions = positions.query("label == 'final_prompt'").rename(columns={"position": "final_position"})
assert not final_positions.duplicated("prompt_id").any()
topk = pd.read_parquet(FULL_RUN / "topk")
topk = topk.query("lens_kind == 'jacobian'").merge(final_positions[["prompt_id", "final_position"]], on="prompt_id", validate="many_to_one")
topk = topk.query("position == final_position").sort_values(["prompt_id", "layer", "rank"])
topk = topk.groupby(["prompt_id", "layer"], group_keys=False).head(DIRECTION_TOP_K)

pair_rows = []
for pair in pairs:
    for side in ("short", "long"):
        prompt = pair[side]
        result = accuracy_by_id.loc[prompt.prompt_id]
        assert int(result.problem_id) == prompt.problem_id
        assert int(result.ctx_size) == nominal_context_size(prompt)
        assert type(result.correct) is bool
        pair_rows.append({
            "problem_id": pair["problem_id"],
            "side": side,
            "prompt_id": prompt.prompt_id,
            "prompt": prompt,
            "prompt_length": int(result.n_input_tokens),
            "correct": bool(result.correct),
        })
pair_info = pd.DataFrame(pair_rows)
pair_info.head()

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
causal_lm.eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
validate_model_lens(model, lens)
LAYERS = tuple(sorted(lens.source_layers))
unembedding_weight = causal_lm.get_output_embeddings().weight
assert set(LAYERS) == set(topk.layer.unique())
print(f"pairs={len(pairs)}, layers={len(LAYERS)}, directions are saved final-position top-{DIRECTION_TOP_K} tokens")

## Record activations and calculate coefficients

The full run saves the prompt/position/direction universe but not hidden-state tensors. This cell records each selected prompt once, at every fitted layer, and immediately applies the project's existing pseudoinverse coordinate calculation.

In [ ]:
prompt_by_id = {row.prompt_id: row.prompt for row in pair_rows}
activation_by_id = {}
for prompt_id, prompt in tqdm(prompt_by_id.items(), desc="Recording matched prompts"):
    prepared = prepare_prompt(prompt, tokenizer, max_seq_len=MAX_SEQ_LEN)
    saved_position = int(final_positions.set_index("prompt_id").loc[prompt_id, "final_position"])
    assert prepared.positions["final_prompt"] == (saved_position,)
    input_ids = torch.tensor([prepared.input_ids], device=context.device)
    with torch.inference_mode(), ActivationRecorder(model.layers, at=LAYERS) as recorder:
        causal_lm(input_ids=input_ids, use_cache=False)
    activation_by_id[prompt_id] = {
        layer: recorder.activations[layer].detach()[:, saved_position, :].float().cpu()
        for layer in LAYERS
    }

rows_out = []
for pair in tqdm(pairs, desc="Building coefficient rows"):
    short_id, long_id = pair["short"].prompt_id, pair["long"].prompt_id
    short_meta = accuracy_by_id.loc[short_id]
    long_meta = accuracy_by_id.loc[long_id]
    transition = f"{'correct' if short_meta.correct else 'wrong'}_{'correct' if long_meta.correct else 'wrong'}"
    pair_ids = (short_id, long_id)
    for layer in LAYERS:
        direction_ids = sorted(set(topk[topk.prompt_id.isin(pair_ids) & (topk.layer == layer)].token_id))
        vectors = torch.stack([
            jlens_vector(lens, unembedding_weight, layer=layer, token_id=int(token_id)).cpu()
            for token_id in direction_ids
        ], dim=-1)
        short_coefficients = lens_coordinates(activation_by_id[short_id][layer], vectors)[0].tolist()
        long_coefficients = lens_coordinates(activation_by_id[long_id][layer], vectors)[0].tolist()
        for token_id, short_coefficient, long_coefficient in zip(direction_ids, short_coefficients, long_coefficients, strict=True):
            delta = long_coefficient - short_coefficient
            rows_out.append({
                "problem_id": pair["problem_id"],
                "short_prompt_id": short_id,
                "long_prompt_id": long_id,
                "short_prompt_length": int(short_meta.n_input_tokens),
                "long_prompt_length": int(long_meta.n_input_tokens),
                "short_correct": bool(short_meta.correct),
                "long_correct": bool(long_meta.correct),
                "correctness_transition": transition,
                "layer": int(layer),
                "direction_token_id": int(token_id),
                "direction_token": tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False),
                "short_coefficient": float(short_coefficient),
                "long_coefficient": float(long_coefficient),
                "delta": float(delta),
                "abs_delta": float(abs(delta)),
            })
asset = pd.DataFrame(rows_out)
asset.head()

In [ ]:
assert len(pairs) == EXPECTED_PAIR_COUNT
assert set(asset.layer) == set(LAYERS)
assert asset[["problem_id", "layer", "direction_token_id"]].duplicated().sum() == 0
assert (asset.short_prompt_id != asset.long_prompt_id).all()
assert (asset.short_prompt_length < asset.long_prompt_length).all()
assert asset.correctness_transition.isin({"correct_correct", "correct_wrong", "wrong_correct", "wrong_wrong"}).all()
coefficient_columns = ["short_coefficient", "long_coefficient", "delta", "abs_delta"]
assert asset[coefficient_columns].notna().all().all()
assert np.isfinite(asset[coefficient_columns].to_numpy()).all()
assert asset.groupby(["problem_id", "layer"]).direction_token_id.nunique().gt(0).all()

transition_counts = asset[["problem_id", "correctness_transition"]].drop_duplicates().correctness_transition.value_counts().sort_index()
print("prompt pairs:", len(pairs))
print("layers:", len(LAYERS), sorted(LAYERS))
print("rows:", len(asset))
display(transition_counts.rename("pair_count").to_frame())
display(asset[["delta", "abs_delta"]].describe())
display(asset.sample(min(10, len(asset)), random_state=1729))

# Direct checks against the original normalized prompt objects and scored results.
for pair in pairs[:3]:
    for prompt in (pair["short"], pair["long"]):
        saved = accuracy_by_id.loc[prompt.prompt_id]
        assert prompt.problem_id == int(saved.problem_id)
        assert prompt.label == bool(saved.label)
        assert evaluate_paper_binary(saved.generated_text, expected=prompt.label).correct == bool(saved.correct)
    pair_asset = asset[asset.problem_id == pair["problem_id"]]
    assert set(pair_asset.short_prompt_id) == {pair["short"].prompt_id}
    assert set(pair_asset.long_prompt_id) == {pair["long"].prompt_id}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pq.write_table(pa.Table.from_pandas(asset, preserve_index=False), ASSET_PATH, compression="zstd")
transition_counts.rename("pair_count").reset_index().to_csv(SUMMARY_PATH, index=False)
print(f"Saved full asset: {ASSET_PATH}")
print(f"Saved transition summary: {SUMMARY_PATH}")